In [6]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import os

In [7]:
# --- Portable Path Setup ---
# 1. Get the directory where the notebook is currently running
current_dir = Path(os.getcwd())

# 2. Find the project root (traverses up until it finds the folder containing 'src')
try:
    project_root = next(p for p in current_dir.parents if (p / 'src').exists() or p.name == 'PedSimCity')
except StopIteration:
    # Fallback to current directory if 'src' isn't found in parents
    project_root = current_dir

# 3. Define the relative path to your census file
file_path = project_root / "src" / "main" / "resources" / "TorinoCentre" / "Torino_censusData.gpkg"

print(f"Attempting to read file at: {file_path}")

try:
    # Read the GeoPackage file
    # Pathlib objects work directly with gpd.read_file
    gdf = gpd.read_file(file_path)

    # Inspect the columns 
    print("\nColumns available:")
    print(gdf.columns.tolist())

    # CRS Check
    print(f"\nCoordinate Reference System: {gdf.crs}")

    # Preview the data
    print("\nFirst 5 rows of data:")
    print(gdf.head())

except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}. Check your folder structure.")
except Exception as e:
    print(f"Error reading the file: {e}")

Attempting to read file at: c:\Users\sgabalog\Documents\PedSim\Working_Version_Connected_to_Gab\PedSimCity\src\main\resources\TorinoCentre\Torino_censusData.gpkg

Columns available:
['COD_REG', 'COD_UTS', 'PRO_COM', 'SEZ21', 'censusZoneID', 'censusZoneTypeID', 'areaType', 'areaID', 'COM_ASC1', 'COM_ASC2', 'COM_ASC3', 'population', 'families', 'residentialUnits', 'residentialBuildings', 'PROCOM', 'SEZ21_ID', 'P1', 'P2', 'P3', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40', 'P41', 'P42', 'P43', 'P44', 'P45', 'P67', 'P68', 'P69', 'P70', 'P71', 'P72', 'P73', 'P74', 'P75', 'P76', 'P77', 'P78', 'P79', 'P80', 'P81', 'P82', 'P83', 'P84', 'P85', 'P86', 'P87', 'P88', 'P89', 'P90', 'P91', 'P92', 'P93', 'P94', 'P95', 'P96', 'P97', 'P98', 'P99', 'P100', 'P101', 'P102', 'P103', 'IT1', 'IT2', 'IT3', 'IT4', 'IT5', 'IT6', 'IT7', 'IT8', 'IT9', 'IT10', 'IT11', 'IT12

In [8]:
# Depending on census data column names will change
# 'P1' here is the population per zone
# Calculate the % of total city residents that live in each zone
total_city_population = gdf['P1'].sum()
gdf['zone_residence_pct'] = gdf['P1'] / total_city_population


# Preview the results
print(gdf[['SEZ21_ID', 'P1', 'zone_residence_pct']].head())

       SEZ21_ID    P1  zone_residence_pct
0           NaN   NaN                 NaN
1           NaN   NaN                 NaN
2           NaN   NaN                 NaN
3  1.272001e+10  20.0            0.000024
4  1.272001e+10   0.0            0.000000


In [9]:
# Remove all rows where zone_residence_pct is 0 as agents cannot spawn there
gdf = gdf[gdf['zone_residence_pct'] > 0].copy()

In [12]:
current_dir = Path(os.getcwd())
try:
    project_root = next(p for p in current_dir.parents if (p / 'src').exists() or p.name == 'PedSimCity')
except StopIteration:
    project_root = current_dir

# 1. Define the columns you want to keep
columns_to_save = ['SEZ21_ID', 'zone_residence_pct', 'geometry']

# 2. Create a new GeoDataFrame containing only those columns
zone_percentages = gdf[columns_to_save].copy()

# 3. Define the output path dynamically
output_path = project_root / "src" / "main" / "resources" / "TorinoCentre" / "TorinoCentre_censusData.gpkg"

try:
    # 4. Save to GPKG
    # Ensure the directory exists
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Save the specific layer
    zone_percentages.to_file(output_path)
    print(f"Successfully: {output_path}")

except Exception as e:
    print(f"An error occurred while saving: {e}")

Successfully: c:\Users\sgabalog\Documents\PedSim\Working_Version_Connected_to_Gab\PedSimCity\src\main\resources\TorinoCentre\TorinoCentre_censusData.gpkg
